# 08 SEEDS: sensitivity of the classification results to the random seed

Repeats the classification with five seeds fixed in advance (42, 1, 2, 3, 4), so that the spread due to the seed can be set against the differences the paper interprets.

**Part 1, pooled target.** All three models under the five sampling strategies at the four windows (D5, W5, W60, W300). Logistic Regression under default weighting has no random component and is fitted once.

**Part 2, other messages.** The messages other than the blower supervision message, built as in `11_CHECK_BLOWER`, for all three models under default weighting at the four windows: XGBoost and CatBoost with five seeds, Logistic Regression once.

**Long run.** Each fit is appended to the results file as soon as it finishes. If the run is interrupted, run the notebook again from the top: completed fits are skipped. W60 and W300 run first. The model code is the same as in `07_CLASSIFY`.

Paper: Sections 2.7.1 and 3.2; Tables 6 to 8.

In [ ]:
import os, sys, time, warnings
import numpy as np, pandas as pd
import sklearn, xgboost, catboost, imblearn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
warnings.filterwarnings("ignore")

A2 = "../ANALYSIS2"
env = os.environ.get
CFG = dict(
    labelled   = os.environ.get("REV_BUILDDIR", "build") + "/telemetry_labelled_v4.pkl",
    episodes   = os.environ.get("REV_BUILDDIR", "build") + "/fault_episodes_v4.csv",
    ref_pooled = f"{A2}/OUTPUT/classification_results_v3.csv",
    ref_groups = f"{A2}/OUTPUT/blower_performance_check.csv",
    outdir     = env("REV_OUTDIR", "OUTPUT") + "/v4",
    blower     = "AUX CAP MACH RM BLOW1 NOT OFF",
    horizons   = env("REV_HORIZONS", "W60,W300,W5,D5").split(","),   # run order
    seeds      = [int(s) for s in env("REV_SEED_LIST", "42,1,2,3,4").split(",")],
    samplers   = env("REV_SAMPLERS", "Default (class weight),Undersampling,Random Oversampling,SMOTE,ADASYN").split(","),
    windows    = {"D5": (-5, 5), "W5": (-5, -1), "W60": (-60, -1), "W300": (-300, -1)},
    patience   = 100, test_frac = 0.20, val_frac = 0.20,
    recall_points = [0.25, 0.50],
)
CFG["results"] = f"{CFG['outdir']}/rev_seeds_results.csv"
NUMERIC = ["KMH", "DIST_M", "KV", "AMP", "KWH", "BV", "PRESSURE", "TRACTION_PCT", "BG1_KN", "BG2_KN", "BG1_DCLV", "BG2_DCLV"]
WHEEL_SLIP = ["Wheel_Slip_TM2", "Wheel_Slip_TM3", "Wheel_Slip_TM4", "Wheel_Slip_TM5", "Wheel_Slip_TM6"]
PRIMARY = NUMERIC + WHEEL_SLIP
os.makedirs(CFG["outdir"], exist_ok=True)
print("Python", sys.version.split()[0], "| pandas", pd.__version__, "| sklearn", sklearn.__version__,
      "| xgboost", xgboost.__version__, "| catboost", catboost.__version__, "| imblearn", imblearn.__version__)
print("Horizons", CFG["horizons"], "| seeds", CFG["seeds"])

## 1. Model code

`three_way_split`, `make_model`, `resample`, `fit_two_stage`, `score` and `evaluate` are the functions of `07_CLASSIFY`; `coverage` is that of `11_CHECK_BLOWER`.

In [ ]:
def chrono_cut(y, frac):
    n = len(y)
    i = int(n * (1 - frac))
    while i < n - 1 and y[i] == 1 and y[i - 1] == 1:
        i += 1
    return i

def three_way_split(y, cfg):
    n = len(y)
    t = int(n * (1 - cfg["test_frac"]))
    while t < n - 1 and y[t] == 1:
        t += 1
    v = chrono_cut(y[:t], cfg["val_frac"])
    return np.arange(v), np.arange(v, t), np.arange(t), np.arange(t, n)

SAMPLERS = {
    "Default (class weight)": None,
    "Undersampling":          lambda s: RandomUnderSampler(random_state=s),
    "Random Oversampling":    lambda s: RandomOverSampler(random_state=s),
    "SMOTE":                  lambda s: SMOTE(random_state=s),
    "ADASYN":                 lambda s: ADASYN(random_state=s),
}

def make_model(name, seed, balanced, y, n_iter=None):
    if name == "Logistic Regression":
        return LogisticRegression(max_iter=2000, random_state=seed, class_weight="balanced" if balanced else None)
    if name == "XGBoost":
        spw = float((y == 0).sum() / max((y == 1).sum(), 1)) if balanced else 1.0
        kw = dict(n_estimators=600, early_stopping_rounds=CFG["patience"]) if n_iter is None \
             else dict(n_estimators=n_iter)
        return XGBClassifier(learning_rate=0.05, max_depth=6, subsample=0.8,
                             colsample_bytree=0.8, eval_metric="aucpr",
                             tree_method="hist", scale_pos_weight=spw,
                             random_state=seed, n_jobs=-1, **kw)
    if name == "CatBoost":
        kw = dict(iterations=800, early_stopping_rounds=CFG["patience"]) if n_iter is None \
             else dict(iterations=n_iter)
        return CatBoostClassifier(learning_rate=0.05, depth=6, eval_metric="PRAUC", auto_class_weights="Balanced" if balanced else None, random_seed=seed, verbose=0, **kw)

def resample(sampler, X, y, seed):
    return (X, y) if sampler is None else SAMPLERS[sampler](seed).fit_resample(X, y)

def fit_two_stage(name, sampler, Xa, ya, Xv, yv, Xtr, ytr, seed):
    balanced = sampler is None
    if name == "Logistic Regression":
        sc = StandardScaler().fit(Xtr)
        Xs, ys = resample(sampler, sc.transform(Xtr), ytr, seed)
        m = make_model(name, seed, balanced, ys).fit(Xs, ys)
        return m, sc, None
    Xs, ys = resample(sampler, Xa, ya, seed)
    m1 = make_model(name, seed, balanced, ys)
    if name == "XGBoost":
        m1.fit(Xs, ys, eval_set=[(Xv, yv)], verbose=False)
        n_iter = int(m1.best_iteration) + 1
    else:
        m1.fit(Xs, ys, eval_set=(Xv, yv), use_best_model=True)
        n_iter = int(m1.get_best_iteration()) + 1
    Xs, ys = resample(sampler, Xtr, ytr, seed)
    m2 = make_model(name, seed, balanced, ys, n_iter=n_iter).fit(Xs, ys)
    return m2, None, n_iter

def score(m, sc, X):
    return m.predict_proba(sc.transform(X) if sc is not None else X)[:, 1]

def evaluate(y, s, recall_points):
    p, r, _ = precision_recall_curve(y, s)
    out = dict(pr_auc=average_precision_score(y, s))
    for rp in recall_points:
        out[f"precision_at_recall_{rp:.2f}"] = float(p[r >= rp].max())
    return out

def coverage(ts, onsets, lo, hi):
    d = np.zeros(len(ts) + 1, dtype=np.int32)
    o = pd.DatetimeIndex(onsets).values.astype("datetime64[s]")
    a = np.searchsorted(ts, o + np.timedelta64(lo, "s"), side="left")
    b = np.searchsorted(ts, o + np.timedelta64(hi, "s"), side="right")
    np.add.at(d, a, 1); np.add.at(d, b, -1)
    return np.cumsum(d)[:-1]

## 2. Data and targets

In [ ]:
tel = pd.read_pickle(CFG["labelled"]).sort_values("DATETIME").reset_index(drop=True)
if env("REV_SAMPLE"):                     # trial runs only
    tel = tel.iloc[:int(len(tel) * float(env("REV_SAMPLE")))].copy()
    print(f"TRIAL RUN on {len(tel):,} rows")
ts = tel["DATETIME"].values.astype("datetime64[s]")
ep = pd.read_csv(CFG["episodes"], parse_dates=["onset", "recovery"])
ep = ep[(ep.onset >= tel.DATETIME.min()) & (ep.onset <= tel.DATETIME.max())]
is_b = ep.fault.eq(CFG["blower"]).values
X_all = tel[PRIMARY].astype("float32").fillna(0).values

def target(h, kind):
    y0 = tel[h].astype(int).values
    if kind == "pooled":
        return y0
    lo, hi = CFG["windows"][h]
    cov_b = coverage(ts, ep.onset[is_b], lo, hi)
    cov_o = coverage(ts, ep.onset[~is_b], lo, hi)
    return np.where((y0 == 1) & (cov_b > 0) & (cov_o == 0), -1, y0)   # other labels

print(f"{len(tel):,} rows; {len(ep)} onsets ({is_b.sum()} blower, {(~is_b).sum()} other)")

## 3. Fits

The job list is built in run order (W60 and W300 first; seed 42 first within each
window), and any fit already in the results file is skipped. Each fit is written to the
file as soon as it is scored.

In [ ]:
jobs = []
for h in CFG["horizons"]:
    for seed in CFG["seeds"]:
        for s in CFG["samplers"]:
            for m in ["XGBoost", "CatBoost"]:
                jobs.append(("pooled", h, s, m, seed))
        for m in ["XGBoost", "CatBoost"]:
            jobs.append(("other labels", h, "Default (class weight)", m, seed))
    jobs.append(("other labels", h, "Default (class weight)", "Logistic Regression", CFG["seeds"][0]))
# Logistic Regression on the pooled target (Tables 6 to 8), run last so the boosted models finish first
for h in CFG["horizons"]:
    for seed in CFG["seeds"]:
        for s in CFG["samplers"]:
            if s.startswith("Default") and seed != CFG["seeds"][0]:
                continue      # deterministic under default weighting
            jobs.append(("pooled", h, s, "Logistic Regression", seed))

KEY = ["target", "horizon", "sampling", "model", "seed"]
done = set()
if os.path.exists(CFG["results"]):
    prev = pd.read_csv(CFG["results"])
    done = set(map(tuple, prev[KEY].astype(str).values))
todo = [j for j in jobs if tuple(map(str, j)) not in done]
print(f"{len(jobs)} fits in total, {len(jobs) - len(todo)} already done, {len(todo)} to run")

In [ ]:
cache = {}
t_all = time.time()
for k, (kind, h, sname, mname, seed) in enumerate(todo, 1):
    if (kind, h) not in cache:
        cache.clear()
        y = target(h, kind)
        keep = y != -1
        X, y = X_all[keep], y[keep]
        cache[(kind, h)] = (X, y, three_way_split(y, CFG))
    X, y, (ia, iv, itr, ite) = cache[(kind, h)]
    t0 = time.time()
    try:
        m, sc, n_iter = fit_two_stage(mname, None if sname.startswith("Default") else sname, X[ia], y[ia], X[iv], y[iv], X[itr], y[itr], seed)
        r = evaluate(y[ite], score(m, sc, X[ite]), CFG["recall_points"])
        status = "ok"
    except Exception as e:
        r, n_iter, status = dict(pr_auc=np.nan), None, f"FAILED: {e}"
    base = y[ite].mean()
    r.update(target=kind, horizon=h, sampling=sname, model=mname, seed=seed, n_iter=n_iter, baseline=base, lift=r["pr_auc"] / base, seconds=round(time.time() - t0, 1), status=status)
    pd.DataFrame([r]).to_csv(CFG["results"], mode="a", index=False, header=not os.path.exists(CFG["results"]))
    print(f"[{k}/{len(todo)}] {kind:12s} {h:5s} {sname:22s} {mname:19s} seed {seed:2d}  "
          f"PR-AUC {r['pr_auc']:.4f}  iters {n_iter}  {r['seconds']:.0f} s", flush=True)
print(f"Finished in {(time.time() - t_all) / 60:.1f} min")

## 4. Comparison with the submitted results (seed 42)

Printed for reference only. The results of the originally submitted manuscript rest on an earlier version of the fault log, so differences are expected and the check does not stop the notebook.

In [ ]:
res = pd.read_csv(CFG["results"])
res = res[res.status == "ok"]
s42 = res[res.seed == 42]
ref_p = pd.read_csv(CFG["ref_pooled"]).set_index(["horizon", "sampling", "model"])["pr_auc"]
ref_g = pd.read_csv(CFG["ref_groups"]).query("variant == 'Without blower'").set_index("horizon")["pr_auc"]

chk = []
for _, r in s42.iterrows():
    if r.target == "pooled":
        ref = ref_p.get((r.horizon, r.sampling, r.model), np.nan)
        ok = abs(r.pr_auc - ref) < 1e-9
    elif r.model == "XGBoost":
        ref = ref_g.get(r.horizon, np.nan)
        ok = round(r.pr_auc, 3) == ref
    else:
        continue
    chk.append(dict(target=r.target, horizon=r.horizon, sampling=r.sampling, model=r.model,
                    pr_auc=round(r.pr_auc, 4), reference=round(ref, 4), reproduced=ok))
chk = pd.DataFrame(chk)
print(chk.to_string(index=False))
if True:   # corrected log: comparison with the submitted results only
    print("corrected log: results of the submitted manuscript shown for comparison only")
else:
    assert chk.reproduced.all(), "seed 42 does not reproduce the submitted figures; stop and investigate"

## 5. Spread across seeds

In [ ]:
g = res.groupby(["target", "horizon", "sampling", "model"])
summ = g.agg(seeds=("seed", "nunique"), mean=("pr_auc", "mean"), sd=("pr_auc", "std"),
             min=("pr_auc", "min"), max=("pr_auc", "max"),
             lift_mean=("lift", "mean"), iters_min=("n_iter", "min"), iters_max=("n_iter", "max"))
summ["range"] = summ["max"] - summ["min"]
summ.to_csv(f"{CFG['outdir']}/rev_seeds_summary.csv")
hz = [h for h in ["D5", "W5", "W60", "W300"] if h in res.horizon.unique()]

for kind in ["pooled", "other labels"]:
    t = summ.loc[kind] if kind in summ.index.get_level_values(0) else None
    if t is None: continue
    cell = t.apply(lambda r: f"{r['mean']:.3f} ± {0 if pd.isna(r['sd']) else r['sd']:.3f}", axis=1)
    print(f"--- {kind}: mean ± SD of PR-AUC over seeds")
    print(cell.unstack("horizon").reindex(columns=hz).to_string(), "\n")

## 6. Is CatBoost's instability across sampling strategies real?

For each model, window and seed, the range of PR-AUC across the five sampling strategies
is compared with the range across seeds for a fixed strategy. If the spread across
strategies is much larger than the spread across seeds, the strategy effect is real; if
the two are similar, it is noise from the seed and the stopping rule.

In [ ]:
p = res[res.target == "pooled"]
pb = p[p.model != "Logistic Regression"]   # boosted models only: iteration counts
across_strat = (pb.groupby(["model", "horizon", "seed"])["pr_auc"].agg(lambda s: s.max() - s.min()).groupby(["model", "horizon"]).mean().rename("range across strategies (mean over seeds)"))
across_seed = (pb.groupby(["model", "horizon", "sampling"])["pr_auc"].agg(lambda s: s.max() - s.min()).groupby(["model", "horizon"]).mean().rename("range across seeds (mean over strategies)"))
iters = (pb.groupby(["model", "horizon"])["n_iter"].agg(lambda s: f"{int(s.min())}–{int(s.max())}").rename("iterations selected"))
early = (pb.assign(e=pb.n_iter <= 5).groupby(["model", "horizon"])["e"].mean().mul(100).round(1).rename("fits stopped within 5 iterations (%)"))
inst = pd.concat([across_strat, across_seed, iters, early], axis=1).reindex(hz, level=1)
print(inst.round(3).to_string())
inst.to_csv(f"{CFG['outdir']}/rev_seeds_instability.csv")

## 7. Ranking stability

For each seed, the best model and sampling strategy at each window, and how often the seed-42 choice remains best.

In [ ]:
best = (p.sort_values("pr_auc", ascending=False).groupby(["horizon", "seed"]).first().reset_index()[["horizon", "seed", "model", "sampling", "pr_auc"]])
best["pr_auc"] = best.pr_auc.round(3)
print(best.pivot(index="seed", columns="horizon", values="model").reindex(columns=hz).to_string(), "\n")
print(best.pivot(index="seed", columns="horizon", values="sampling").reindex(columns=hz).to_string(), "\n")
print(best.pivot(index="seed", columns="horizon", values="pr_auc").reindex(columns=hz).to_string())
best.to_csv(f"{CFG['outdir']}/rev_seeds_best.csv", index=False)

___
**END**